# CS5090 Project 1 — Phase 1 training on Kaggle
Settings: Accelerator = GPU (T4 x1 or P100), Internet = ON, Persistence = Files. Set `RUN` below: `baseline` first; `ablation_nopos` only AFTER `PREDICTION.md` is committed.

In [ ]:
RUN = 'baseline'   # or 'ablation_nopos'
REPO_URL = 'https://github.com/JabobleHead/CS5090-Project-1.git'
COMMIT = ''  # optional pinned commit SHA for exact reproduction

In [ ]:
import subprocess, os
subprocess.run(f'git clone {REPO_URL} /kaggle/working/repo', shell=True, check=True)
if COMMIT: subprocess.run(f'git -C /kaggle/working/repo checkout {COMMIT}', shell=True, check=True)
os.chdir('/kaggle/working/repo')
!pip install -q tokenizers datasketch

In [ ]:
# Rebuild the data exactly as the repo documents (~10 min). Skip if data/code/*.bin already exists (e.g. from a Kaggle dataset).
if not os.path.exists('data/code/train.bin'):
    !python scripts/collect.py && python scripts/filter_dedup.py && python scripts/train_tokenizer.py && python scripts/prepare.py

In [ ]:
import torch, sys, platform, json
print(torch.__version__, torch.cuda.get_device_name(0), 'bf16 supported:', torch.cuda.is_bf16_supported())
# 200-iter smoke test: loss should fall from ~9.0 (= ln 8192) and memory should fit
!python nanogpt/train.py configs/{RUN}.py --max_iters=200 --lr_decay_iters=200 --eval_interval=100 --out_dir=out-smoke --log_csv=results/smoke.csv

In [ ]:
# FULL RUN (resume-able: if the session dies, re-run with --init_from=resume)
!rm -f results/{RUN}_loss.csv
!python nanogpt/train.py configs/{RUN}.py --compile=False 2>&1 | tee results/{RUN}_train.log

In [ ]:
# record run metadata for the report, then download out-*/ckpt.pt and results/ (Kaggle: Output tab or zip below)
meta = dict(run=RUN, torch=torch.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0), python=sys.version, platform=platform.platform(), repo_commit=subprocess.getoutput('git rev-parse HEAD'))
json.dump(meta, open(f'results/{RUN}_run_metadata.json','w'), indent=2); print(meta)
!zip -qr /kaggle/working/{RUN}_outputs.zip out-{RUN.replace('_','-')}* results